In [1]:
import pandas as pd
import numpy as np
import unicodedata 
import re


In [2]:
CAPES = pd.read_excel('dados/DAV_ResultadosQuadrienal2025.xlsx')
# imprimir colunas para debug
print("Colunas CAPES:", CAPES.columns.tolist())

#
# encontrando categoria certa
#
possiveis = ['SG_IES','SIGLA_IES','SIGLA IES','SIGLA','IES']
col_encontrada = None
cols_norm = {c: c.upper().replace(' ', '_') for c in CAPES.columns}
for nome in possiveis:
    for orig, norm in cols_norm.items():
        if norm == nome.upper().replace(' ', '_'):
            col_encontrada = orig
            break
    if col_encontrada:
        break

if col_encontrada is None:
    raise KeyError(f"Nenhuma das colunas esperadas encontrada. Colunas disponíveis: {CAPES.columns.tolist()}")
else:
    CAPES['sigla_ies_clean'] = CAPES[col_encontrada].astype(str).str.upper().str.strip()

    import pandas as pd

#
# Carregando os dados do CNPq e da CAPES
# 
cnpq = pd.read_parquet('dados/cnpqBolsasAuxilios.parquet')
CAPES = pd.read_excel('dados/DAV_ResultadosQuadrienal2025.xlsx')

#
# Coluna Nota seja inteiro 
# 
CAPES['Nota'] = pd.to_numeric(CAPES['Nota'], errors='coerce').astype('Int64')

#
# Agrupar pela Sigla da IES 
# 
capes_nota = CAPES.groupby('Sigla IES')['Nota'].max().reset_index() 
cnpq['sigla_cnpq_clean'] = cnpq['sigla_instituicao_destino'].astype(str).str.upper().str.strip()
capes_nota['sigla_capes_clean'] = capes_nota['Sigla IES'].astype(str).str.upper().str.strip()

# merge
df_final = pd.merge(
    cnpq,
    capes_nota[['sigla_capes_clean', 'Nota']],
    left_on='sigla_cnpq_clean',
    right_on='sigla_capes_clean',
    how='left'
)

df_final = df_final.rename(columns={'Nota': 'capes_nota_ies'})
df_final['capes_nota_ies'] = df_final['capes_nota_ies'].fillna(0)
df_final = df_final.drop(columns=['sigla_cnpq_clean', 'sigla_capes_clean'])
print(df_final[['sigla_instituicao_destino', 'capes_nota_ies']].head())

df_final.to_parquet('dados/cnpq_com_capes.parquet', index=False)

Colunas CAPES: ['Código IES', 'Sigla IES', 'Instituição de Ensino', 'Status Jurídico', 'Categoria Administrativa', 'Confessional Comunitária', 'Organização Acadêmica', 'Área de Avaliação', 'Colégio', 'Código do Programa', 'Nome do Programa', 'Região', 'UF', 'Grau Acadêmico', 'Nota', 'Código IES Nacional', 'Sigla IES Nacional', 'IES Nacional']
  sigla_instituicao_destino  capes_nota_ies
0                      UFMG               7
1               SENAI/DR/BA               0
2                       USP               7
3                      UFMG               7
4                       USP               7
